# 02 — MLflow Tracing with Gemini

**UI tab:** Traces

**Tracing** records every LLM call as a structured timeline — inputs, outputs, duration, and nested steps (called **spans**).

| Method | How |
|---|---|
| Auto-trace everything | `mlflow.gemini.autolog()` |
| Trace a function | `@mlflow.trace` decorator |
| Custom named spans | `mlflow.start_span()` context manager |

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai --quiet

In [ ]:
import os
from google import genai
from google.genai import types
import mlflow

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("02-MLflow-Tracing")

print("MLflow", mlflow.__version__, "ready")

## Method 1 — Autolog (zero code changes)

In [ ]:
# One call — every Gemini request is automatically traced from now on
mlflow.gemini.autolog()
print("Autolog ON")

In [ ]:
# This plain Gemini call is captured automatically as a trace
with mlflow.start_run(run_name="autolog-demo"):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=["What is MLflow tracing? Answer in 2 sentences."]
    )
    print(response.text)

print("\nOpen MLflow UI → Traces tab → you will see a new trace with the full request/response.")

## Method 2 — `@mlflow.trace` decorator

In [ ]:
@mlflow.trace(name="ask-gemini", span_type="LLM")
def ask_gemini(question: str) -> str:
    return client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[question]
    ).text

@mlflow.trace(name="format-output", span_type="UNKNOWN")
def format_output(text: str) -> str:
    return f"[{len(text.split())} words] {text.strip()}"

# Each decorated function appears as a named span in the trace tree
with mlflow.start_run(run_name="decorator-trace"):
    raw = ask_gemini("What is an MLflow experiment?")
    result = format_output(raw)
    print(result)

## Method 3 — `mlflow.start_span()` for nested steps

In [ ]:
# Simulate a simple RAG pipeline: retrieve → generate
# Each step is a separate span visible in the trace timeline

DOCS = {"mlflow": "MLflow manages the ML lifecycle: tracking, registry, and deployment."}

with mlflow.start_run(run_name="rag-pipeline"):

    with mlflow.start_span(name="rag-pipeline", span_type="CHAIN") as root:
        query = "What does MLflow do?"
        root.set_attribute("query", query)

        # Step 1: retrieve
        with mlflow.start_span(name="retrieve", span_type="RETRIEVER") as s:
            context = DOCS.get("mlflow", "")
            s.set_attribute("context", context)

        # Step 2: generate
        with mlflow.start_span(name="generate", span_type="LLM") as s:
            prompt = f"Context: {context}\n\nQuestion: {query}"
            answer = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[prompt]
            ).text
            s.set_attribute("answer", answer[:200])

        root.set_attribute("final_answer", answer[:200])

print(f"Answer: {answer}")
print("\nTraces tab → expand the trace → see retrieve and generate spans side by side.")

## MLflow UI — What to explore
```
Traces tab
├── autolog-demo   → click → see auto-captured request/response
├── decorator-trace → click → see ask-gemini + format-output spans
└── rag-pipeline   → click → expand to see retrieve → generate hierarchy
```
**Next →** `03_mlflow_sessions_chat.ipynb`